# Score the V9 robustness sets with the two trained LoRA models (Qwen3.5-9B and Gemma-4-12B-QAT) - Colab

The training sessions are gone, but the adapters were saved. This notebook loads **one model at a time** (base model in 4-bit + your saved adapter, exactly the way it was trained), checks that it reproduces the scores already stored in your main `.npz`, and then scores the texts that V9_2 needs for **Table 1e** (unfiltered OSDG test: label-noise slices) and **Table 1f** (keyword perturbations).

**No training happens here.** Prompt, label words, left padding and the scoring function are copied unchanged from the two training notebooks, so the probabilities are comparable with the ones in `qwen_lora_osdg.npz` / `gemma12b_qat_lora_osdg.npz`.

In [ ]:
!nvidia-smi

Fri Oct 23 10:13:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Install (run once, then Runtime > Restart session)
Same pins as the training notebooks. Dependency-conflict warnings from pip are usually harmless.

In [ ]:
!pip install -q --upgrade unsloth unsloth_zoo "transformers>=5.0.0,<=5.17.0" scikit-learn tqdm

## Extract the uploaded files

In [ ]:
import os, zipfile, glob
assert os.path.exists("qwen_export.zip"), "Drag the qwen_export.zip made by V9_2 into the Files pane. Files here: %s" % os.listdir(".")
zipfile.ZipFile("qwen_export.zip").extractall("qwen_export")
print("export folder:", sorted(os.listdir("qwen_export")))
for need in ["robust_osdg_test_all.parquet", "robust_pert.parquet", "osdg_test.parquet", "meta.json"]:
    assert os.path.exists(f"qwen_export/{need}"), f"{need} is missing in qwen_export.zip: it must come from V9_2 (cell 5b), not from the old training export."
for z in ["qwen_lora_osdg_adapter.zip", "gemma12b_qat_lora_osdg_adapter.zip"]:
    if os.path.exists(z):
        zipfile.ZipFile(z).extractall("adapter_" + z.replace("_adapter.zip", ""))
        print("extracted", z)
    else:
        print("not uploaded (needed only if you run this model):", z)
print("main score files present:", [f for f in ["qwen_lora_osdg.npz", "gemma12b_qat_lora_osdg.npz"] if os.path.exists(f)])

export folder: ['meta.json', 'osdg_dev.parquet', 'osdg_test.parquet', 'osdg_train.parquet', 'robust_osdg_test_all.parquet', 'robust_pert.parquet', 'sdgi_chunks_dev.parquet', 'sdgi_chunks_test.parquet']
extracted qwen_lora_osdg_adapter.zip
extracted gemma12b_qat_lora_osdg_adapter.zip
main score files present: ['qwen_lora_osdg.npz', 'gemma12b_qat_lora_osdg.npz']


In [ ]:
# imports. `unsloth` must be imported BEFORE transformers / peft.
import os, sys, json, time, math, random, gc, hashlib
from pathlib import Path
from unsloth import FastLanguageModel, FastModel
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from google.colab import files

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Config
Everything that differs between the two training notebooks is listed here (loader order, chat template, scoring batch size). The prompt and label words are shared and unchanged.

In [ ]:
RUN = ["qwen", "gemma"]            # models to score, one after the other; finished ones (their *_extra.npz exists) are skipped
# from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # only if the Gemma download asks for a login

SPECS = {
    "qwen": dict(base="unsloth/Qwen3.5-9B", gemma=False, adapter_dir="adapter_qwen_lora_osdg",
                 main_npz="qwen_lora_osdg.npz", stem="qwen_lora_osdg", eval_bs=32, max_tokens=12000),
    "gemma": dict(base="google/gemma-4-12B-it-qat-q4_0-unquantized", gemma=True, adapter_dir="adapter_gemma12b_qat_lora_osdg",
                  main_npz="gemma12b_qat_lora_osdg.npz", stem="gemma12b_qat_lora_osdg", eval_bs=16, max_tokens=6000),
}
MAX_WORDS, MAX_SEQ = 200, 768       # as in training (each text is cut to its first 200 words)
CHECK_N, CHECK_MIN_AGREE = 300, 0.97   # reproduction gate: argmax agreement with the stored scores on 300 random test paragraphs
ALLOW_UNVERIFIED = False           # True only if you cannot upload the main .npz (NOT recommended: the gate is what proves the adapter was loaded correctly)

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU."
print("GPU:", torch.cuda.get_device_name(0), "| VRAM %.1f GiB" % (torch.cuda.get_device_properties(0).total_memory / 2**30))
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
def amp_ctx():
    return torch.autocast("cuda", dtype=AMP, enabled=True)
EXP = Path("qwen_export")
meta = json.load(open(EXP / "meta.json"))
COMMON = [int(s) for s in meta["COMMON"]]; K = len(COMMON)
assert COMMON == list(range(1, 17)), "the label words below assume SDGs 1-16"
te = pd.read_parquet(EXP / "osdg_test.parquet")                       # filtered OSDG test set (used for the reproduction gate)
t_all = pd.read_parquet(EXP / "robust_osdg_test_all.parquet").text.astype(str).tolist()    # unfiltered OSDG test split -> Table 1e
t_pert = pd.read_parquet(EXP / "robust_pert.parquet").text.astype(str).tolist()            # 200 paragraphs x 3 conditions -> Table 1f
md5 = lambda xs: hashlib.md5("\n".join(xs).encode()).hexdigest()
H_ALL, H_PERT = md5(t_all), md5(t_pert)
for k, h in [("all_md5", H_ALL), ("pert_md5", H_PERT)]:
    if k in meta:
        assert meta[k] == h, f"{k} in meta.json does not match the parquet file: the export is inconsistent, re-export from V9_2"
print(f"unfiltered test: {len(t_all)} texts | perturbation set: {len(t_pert)} texts | filtered test (gate): {len(te)} | models to run: {RUN}")

GPU: NVIDIA L4 | VRAM 22.0 GiB
unfiltered test: 5374 texts | perturbation set: 600 texts | filtered test (gate): 3566 | models to run: ['qwen', 'gemma']


## Scorer
One class = one loaded model. The code of `build_prompt`, `encode`, `collate`, `label_logits` and `predict` is copied from the training notebooks (Steps 4 and 5), only wrapped in a class so that two models can be loaded one after the other.
Loading: the base model is loaded in 4-bit by Unsloth exactly like in training, then the saved adapter is attached with PEFT (no merge). If that fails, Unsloth is asked to load the adapter folder directly.

In [ ]:
LABEL_WORDS = ["Poverty", "Hunger", "Health", "Education", "Gender", "Water", "Energy", "Work",
               "Industry", "Inequality", "Cities", "Consumption", "Climate", "Ocean", "Land", "Peace"]
PROMPT = ("Classify the text into the single UN Sustainable Development Goal it mainly addresses. "
          "Answer with exactly one word from this list: " + ", ".join(LABEL_WORDS) + ".\n"
          "(Health = good health and well-being; Gender = gender equality; Water = clean water and sanitation; "
          "Energy = affordable and clean energy; Work = decent work and economic growth; "
          "Industry = industry, innovation and infrastructure; Inequality = reduced inequalities; "
          "Cities = sustainable cities and communities; Consumption = responsible consumption and production; "
          "Ocean = life below water; Land = life on land; Peace = peace, justice and strong institutions.)\n\n"
          "Text: {text}")

class Scorer:
    def __init__(self, spec):
        self.spec = spec
        kw = dict(model_name=spec["base"], max_seq_length=MAX_SEQ, load_in_4bit=True, load_in_16bit=False, full_finetuning=False)
        self.loader, model, tokenizer = None, None, None
        for L in ([FastModel, FastLanguageModel] if spec["gemma"] else [FastLanguageModel, FastModel]):
            try:
                model, tokenizer = L.from_pretrained(**kw); self.loader = L; break
            except Exception as e:
                print(f"{L.__name__} failed ({repr(e)[:250]}) -> trying the other loader")
        if model is None:
            raise RuntimeError("both Unsloth loaders failed: read the messages above")
        tokenizer = self._template(tokenizer)
        ad = spec["adapter_dir"]
        cfg_files = glob.glob(f"{ad}/**/adapter_config.json", recursive=True)
        assert cfg_files, f"no adapter_config.json under {ad}: upload the adapter zip and re-run the extract cell"
        ad = os.path.dirname(cfg_files[0])
        try:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, ad, is_trainable=False); how = "PeftModel.from_pretrained on the freshly loaded base"
        except Exception as e:
            print("PeftModel attach failed (", repr(e)[:250], ") -> loading the adapter folder through Unsloth")
            del model; gc.collect(); torch.cuda.empty_cache()
            model, tokenizer = self.loader.from_pretrained(**{**kw, "model_name": ad}); tokenizer = self._template(tokenizer)
            how = "Unsloth from_pretrained(adapter folder)"
        self.model = model.eval()
        self.tok = getattr(tokenizer, "tokenizer", tokenizer)
        n_lora = sum(p.numel() for n, p in self.model.named_parameters() if "lora_" in n)
        print(f"loaded {spec['base']} + adapter via {how} | loader {self.loader.__name__} | LoRA params {n_lora/1e6:.1f} M | GPU memory {torch.cuda.memory_allocated()/2**30:.1f} GiB")
        assert n_lora > 0, "no LoRA weights found in the model: the adapter was not attached"
        # label tokens (first token of each answer word, as in training)
        base_ids = self.tok(self.build_prompt("x"), add_special_tokens=False)["input_ids"]
        ids = []
        for w in LABEL_WORDS:
            full = self.tok(self.build_prompt("x") + w, add_special_tokens=False)["input_ids"]
            assert full[:len(base_ids)] == base_ids, "prompt tokens changed when the answer was appended"
            ids.append(full[len(base_ids)])
        assert len(set(ids)) == K
        self.label_ids = torch.tensor(ids, device="cuda")
        print("label tokens:", [self.tok.decode([i]) for i in ids])
        print("end of an example prompt:", repr(self.build_prompt("Example text.")[-90:]))
        self.pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id

    def _template(self, tokenizer):
        if self.spec["gemma"]:
            try:
                from unsloth.chat_templates import get_chat_template
                tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
            except Exception as e:
                print("could not set the gemma-4 chat template, using the built-in one:", repr(e)[:150])
        return tokenizer

    def build_prompt(self, text):
        t = " ".join(str(text).split()[:MAX_WORDS])
        msgs = [{"role": "user", "content": PROMPT.format(text=t)}]
        try:
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def encode(self, texts):
        return [self.tok(self.build_prompt(t), add_special_tokens=False)["input_ids"] for t in texts]

    def collate(self, seqs):
        L = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), L), self.pad, dtype=torch.long); att = torch.zeros((len(seqs), L), dtype=torch.long)
        for i, s in enumerate(seqs):
            ids[i, L - len(s):] = torch.tensor(s); att[i, L - len(s):] = 1       # left padding
        return ids.cuda(), att.cuda()

    def label_logits(self, ids, att):
        try:
            out = self.model(input_ids=ids, attention_mask=att, use_cache=False, logits_to_keep=1)
        except TypeError:
            out = self.model(input_ids=ids, attention_mask=att, use_cache=False, num_logits_to_keep=1)
        return out.logits[:, -1, :][:, self.label_ids]

    @torch.no_grad()
    def predict(self, texts, desc="scoring"):
        self.model.eval()
        seqs = self.encode(list(texts)); order = np.argsort([-len(s) for s in seqs]); lens = [len(s) for s in seqs]
        print(f"{desc}: {len(seqs)} texts | longest {max(lens)} tokens | median {int(np.median(lens))}")
        out = np.zeros((len(seqs), K), dtype=np.float32)

        def run(idx):
            ok = False
            try:
                ids, att = self.collate([seqs[j] for j in idx])
                with amp_ctx():
                    lg = self.label_logits(ids, att)
                out[idx] = torch.softmax(lg.float(), dim=-1).cpu().numpy(); ok = True
            except torch.cuda.OutOfMemoryError:
                pass
            if not ok:
                ids = att = lg = None; gc.collect(); torch.cuda.empty_cache()
                if len(idx) == 1:
                    raise RuntimeError("a single sequence does not fit in memory")
                h = len(idx) // 2; run(idx[:h]); run(idx[h:])

        pbar = tqdm(total=len(order), desc=desc); i = 0
        while i < len(order):
            b = max(1, min(self.spec["eval_bs"], self.spec["max_tokens"] // len(seqs[order[i]])))
            idx = order[i:i + b]; run(idx); i += len(idx); pbar.update(len(idx))
        pbar.close(); return out

    def close(self):
        self.model = None; self.label_ids = None
        gc.collect(); torch.cuda.empty_cache()

## Run: reproduction gate, then scoring
For every model in `RUN`: (1) the loaded model scores 300 random paragraphs of the filtered test set and must agree with the scores stored in its main `.npz` (at least 97% identical argmax). If it does not, the adapter or prompt is not the one that produced your results and **nothing is saved** for that model. (2) The unfiltered test set and the perturbation set are scored and saved to `<name>_extra.npz` (with fingerprints of the texts, which V9_2 checks).

In [ ]:
import glob
ERRORS = {}
for key in RUN:
    spec = SPECS[key]; out_path = Path(spec["stem"] + "_extra.npz")
    if out_path.exists():
        print(f"[{key}] {out_path} already exists -> skipped"); continue
    print("=" * 100, f"\n[{key}] {spec['base']}")
    sc = None
    try:
        sc = Scorer(spec)
        # ---- (1) reproduction gate
        gate = None
        if os.path.exists(spec["main_npz"]):
            ref = np.load(spec["main_npz"], allow_pickle=True)["osdg_test"]
            assert len(ref) == len(te), f"{spec['main_npz']} has {len(ref)} test rows but the export has {len(te)}: the export does not match the run that made this file"
            ii = np.sort(np.random.default_rng(0).choice(len(te), CHECK_N, replace=False))
            chk = sc.predict(te.text.iloc[ii].tolist(), desc="reproduction check")
            agree = float((chk.argmax(1) == ref[ii].argmax(1)).mean()); mad = float(np.abs(chk - ref[ii]).mean())
            acc = float((chk.argmax(1) == te.y.values[ii]).mean())
            print(f"[{key}] reproduction: argmax agreement {agree:.3f} | mean |prob diff| {mad:.4f} | accuracy on these paragraphs {acc:.3f}")
            assert agree >= CHECK_MIN_AGREE, "the reloaded model does NOT reproduce the saved scores: nothing is saved for this model (adapter or prompt differs)"
            gate = agree
        elif ALLOW_UNVERIFIED:
            print(f"[{key}] WARNING: {spec['main_npz']} not uploaded, running WITHOUT the reproduction gate")
        else:
            raise FileNotFoundError(f"upload {spec['main_npz']} (needed for the reproduction check) or set ALLOW_UNVERIFIED = True")
        # ---- (2) score the robustness sets (each finished set is also kept on disk)
        t0 = time.time()
        S_pert = sc.predict(t_pert, desc="perturbations"); np.save(f"/content/{spec['stem']}_pert.npy", S_pert); files.download(f"/content/{spec['stem']}_pert.npy")
        S_all = sc.predict(t_all, desc="osdg_test_all"); np.save(f"/content/{spec['stem']}_all.npy", S_all); files.download(f"/content/{spec['stem']}_all.npy")
        assert S_all.shape == (len(t_all), K) and S_pert.shape == (len(t_pert), K)
        np.savez(out_path, osdg_test_all=S_all, pert=S_pert, all_md5=H_ALL, pert_md5=H_PERT, model=spec["base"],
                 repro_agreement=np.nan if gate is None else gate, scoring_time_s=time.time() - t0)
        print(f"[{key}] saved {out_path} | {S_all.shape} + {S_pert.shape} | {(time.time() - t0) / 60:.1f} min")
        files.download(str(out_path))
    except Exception as e:
        ERRORS[key] = repr(e); print(f"[{key}] FAILED: {repr(e)[:400]}")
    finally:
        if sc is not None:
            sc.close()
        del sc; gc.collect(); torch.cuda.empty_cache()
        print(f"[{key}] GPU memory after cleanup: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
print("\nDONE." if not ERRORS else f"\nFINISHED WITH ERRORS: {ERRORS}")
print("files to copy into outputs/preds/:", sorted(glob.glob("*_extra.npz")))

[qwen] unsloth/Qwen3.5-9B
==((====))==  Unsloth 2026.10.7: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen3_5DecoderLayer).
loaded unsloth/Qwen3.5-9B + adapter via PeftModel.from_pretrained on the freshly loaded base | loader FastLanguageModel | LoRA params 29.1 M | GPU memory 8.1 GiB
label tokens: ['Po', 'H', 'Health', 'Education', 'Gender', 'Water', 'Energy', 'Work', 'Industry', 'In', 'Cities', 'Consum', 'Climate', 'Ocean', 'Land', 'Peace']
end of an example prompt: 'ng institutions.)\n\nText: Example text.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
reproduction check: 300 texts | longest 409 tokens | median 270


reproduction check:   0%|          | 0/300 [00:00<?, ?it/s]

`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[qwen] reproduction: argmax agreement 1.000 | mean |prob diff| 0.0003 | accuracy on these paragraphs 0.900
perturbations: 600 texts | longest 401 tokens | median 271


... (23 outputs left out)

[gemma] saved gemma12b_qat_lora_osdg_extra.npz | (5374, 16) + (600, 16) | 20.9 min


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[gemma] GPU memory after cleanup: 0.0 GiB

DONE.
files to copy into outputs/preds/: ['gemma12b_qat_lora_osdg_extra.npz', 'qwen_lora_osdg_extra.npz']
